In [1]:
import pandas as pd

# Inisialisasi dan Pemuatan Data
price_path = '../dataset/dataset_cabe_mb_cleaned.csv'
weather_path = '../dataset/dataset_cuaca_bandung_cleaned.csv'

df_harga = pd.read_csv(price_path)
df_cuaca = pd.read_csv(weather_path)

In [2]:
df_harga.info()

<class 'pandas.DataFrame'>
RangeIndex: 1386 entries, 0 to 1385
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Tanggal            1386 non-null   str    
 1   Cabai Merah Besar  1386 non-null   float64
dtypes: float64(1), str(1)
memory usage: 21.8 KB


In [3]:
df_cuaca.info()

<class 'pandas.DataFrame'>
RangeIndex: 727 entries, 0 to 726
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   TANGGAL  727 non-null    str    
 1   TN       727 non-null    float64
 2   TX       727 non-null    float64
 3   TAVG     727 non-null    float64
 4   RH_AVG   727 non-null    float64
 5   RR       727 non-null    float64
 6   SS       727 non-null    float64
 7   FF_X     727 non-null    float64
 8   DDD_X    727 non-null    float64
dtypes: float64(8), str(1)
memory usage: 51.2 KB


In [4]:
# Standarisasi Kolom Tanggal

# Konversi ke datetime
df_harga['Tanggal'] = pd.to_datetime(df_harga['Tanggal'])
df_cuaca['TANGGAL'] = pd.to_datetime(df_cuaca['TANGGAL'])

# Menyeragamkan nama kolom menjadi 'tanggal'
df_harga = df_harga.rename(columns={'Tanggal': 'tanggal'})
df_cuaca = df_cuaca.rename(columns={'TANGGAL': 'tanggal'})


In [5]:
# Penggabungan Dataset (Inner Join)
df_merged = pd.merge(df_cuaca, df_harga, on='tanggal', how='left')

# Sekarang set_index jika ingin
df_merged.set_index('tanggal', inplace=True)
df_merged.sort_index(inplace=True)
# Validasi
print("\n--- Validasi Kualitas Data ---")
print("Nilai Kosong per Kolom:\n", df_merged.isnull().sum())

if not df_merged.empty:
    print(f"Rentang Data: {df_merged.index.min().date()} s/d {df_merged.index.max().date()}")
    print(f"Total Baris Setelah Digabung: {len(df_merged)}")

else:
    print("Warning: Dataset hasil penggabungan kosong!")


--- Validasi Kualitas Data ---
Nilai Kosong per Kolom:
 TN                     0
TX                     0
TAVG                   0
RH_AVG                 0
RR                     0
SS                     0
FF_X                   0
DDD_X                  0
Cabai Merah Besar    209
dtype: int64
Rentang Data: 2024-05-01 s/d 2026-04-27
Total Baris Setelah Digabung: 727


In [6]:
# Cek kolom yang memiliki setidaknya satu nilai null
df_merged[df_merged.isna().any(axis=1)]

,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X,Cabai Merah Besar
tanggal,,,,,,,,,
2024-05-04,21.4,31.6,25.1,76.0,2.8,6.7,3.0,120.0,NaN
2024-05-05,21.2,31.2,25.5,72.0,1.7,6.4,5.0,220.0,NaN
2024-05-11,21.4,29.6,24.8,76.0,0.1,6.8,2.0,240.0,NaN
2024-05-12,21.6,32.1,26.2,73.0,0.0,3.4,3.0,60.0,NaN
2024-05-18,20.6,30.2,25.2,75.0,16.8,5.5,3.0,241.0,NaN
...,...,...,...,...,...,...,...,...,...
2026-04-18,21.4,29.8,24.8,82.0,5.7,3.7,2.0,270.0,NaN
2026-04-19,21.1,30.6,24.1,80.0,0.0,6.4,3.0,250.0,NaN
2026-04-25,20.8,32.0,25.1,74.0,0.0,7.2,3.0,40.0,NaN


In [7]:
df_clean = df_merged.resample('D').interpolate(method='time').ffill().bfill()

In [8]:
df_clean[df_clean.isna().any(axis=1)]

,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X,Cabai Merah Besar
tanggal,,,,,,,,,


In [9]:
df_clean.tail()

,TN,TX,TAVG,RH_AVG,RR,SS,FF_X,DDD_X,Cabai Merah Besar
tanggal,,,,,,,,,
2026-04-23,21.8,29.2,24.9,85.0,0.2,4.5,2.0,280.0,64600.0
2026-04-24,20.6,31.0,25.0,78.0,0.0,4.5,3.0,270.0,64600.0
2026-04-25,20.8,32.0,25.1,74.0,0.0,7.2,3.0,40.0,64600.0
2026-04-26,20.4,30.4,25.0,77.0,0.0,6.7,4.0,287.0,64600.0
2026-04-27,21.6,31.4,25.4,81.0,3.4,7.2,4.0,290.0,64600.0


In [10]:
df_clean.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 727 entries, 2024-05-01 to 2026-04-27
Freq: D
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   TN                 727 non-null    float64
 1   TX                 727 non-null    float64
 2   TAVG               727 non-null    float64
 3   RH_AVG             727 non-null    float64
 4   RR                 727 non-null    float64
 5   SS                 727 non-null    float64
 6   FF_X               727 non-null    float64
 7   DDD_X              727 non-null    float64
 8   Cabai Merah Besar  727 non-null    float64
dtypes: float64(9)
memory usage: 56.8 KB


In [11]:
# 5. Penyimpanan Hasil Akhir
output_path = '../dataset/dataset_merged.csv'
df_clean.to_csv(output_path)
print(f"\nDataset berhasil disimpan ke {output_path}")


Dataset berhasil disimpan ke ../dataset/dataset_merged.csv
